# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

### Plan - Add set ticket tool, add a new database that stores flight ticket : Source, Destination, Date, Time. Book Ticket, Cancel Ticket, Check Ticket

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content



OpenAI API Key exists and begins sk-proj-


### Step 1: Adding all the necessary functions

In [2]:
price_db = "prices.db"

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(price_db) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    with sqlite3.connect(price_db) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

ticket_DB = "tickets.db"

with sqlite3.connect(ticket_DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS tickets (Unique_ID TEXT PRIMARY KEY, user TEXT, Govt_ID TEXT, source TEXT, destination TEXT, price REAL, date DATE, time TIME)')
    conn.commit()

def book_ticket(user, govt_id, source, destination, date, time):
    print(f"DATABASE TOOL CALLED: Booking ticket for {user} from {source} to {destination} on {date} at {time}", flush=True)

    price = get_ticket_price(destination)
    if "No price data available" in price:
        return "No price data available for the destination city. Cannot book ticket."

    with sqlite3.connect(ticket_DB) as conn:
        cursor = conn.cursor()
        unique_id = f"{user}_{govt_id}_{source}_{destination}_{date}_{time}"
        cursor.execute('INSERT INTO tickets (Unique_ID, user, Govt_ID, source, destination, price, date, time) VALUES (?, ?, ?, ?, ?, ?, ?, ?)', 
                       (unique_id, user, govt_id, source, destination, price, date, time))
        conn.commit()
        return f"Ticket booked successfully for {user} from {source} to {destination} on {date} at {time}. Price: ${price}"

def cancel_ticket(unique_id):
    print(f"DATABASE TOOL CALLED: Cancelling ticket with ID {unique_id}", flush=True)
    with sqlite3.connect(ticket_DB) as conn:
        cursor = conn.cursor()
        cursor.execute('DELETE FROM tickets WHERE Unique_ID = ?', (unique_id,))
        conn.commit()
        return f"Ticket with ID {unique_id} has been cancelled."

def show_bookings(user):
    print(f"DATABASE TOOL CALLED: Showing bookings for user {user}", flush=True)
    with sqlite3.connect(ticket_DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT Unique_ID, source, destination, price, date, time FROM tickets WHERE user = ?', (user,))
        results = cursor.fetchall()
        if not results:
            return "No bookings found for this user."
        
        bookings = []
        for row in results:
            unique_id, source, destination, price, date, time = row
            bookings.append(f"ID: {unique_id}, From: {source}, To: {destination}, Price: ${price}, Date: {date}, Time: {time}")
        
        return "\n".join(bookings)

In [3]:
get_ticket_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_ticket_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city for which to set the ticket price",
            },
            "price": {
                "type": "number",
                "description": "The price to set for the ticket",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

book_ticket_function = {
    "name": "book_ticket",
    "description": "Book a ticket for a user with Government Id from source to destination on a specific date and time.",
    "parameters": {
        "type": "object",
        "properties": {
            "user": {
                "type": "string",
                "description": "The name of the user booking the ticket",
            },
            "govt_id": {
                "type": "string",
                "description": "Government ID of the user for verification",
            },
            "source": {
                "type": "string",
                "description": "The source city for the ticket",
            },
            "destination": {
                "type": "string",
                "description": "The destination city for the ticket",
            },
            "date": {
                "type": "string",
                "description": "The date of travel (YYYY-MM-DD)",
            },
            "time": {
                "type": "string",
                "description": "The time of travel (HH:MM)",
            },
        },
        "required": ["user", "govt_id", "source", "destination", "date", "time"],
        "additionalProperties": False
    }
}

cancel_ticket_function = {
    "name": "cancel_ticket",
    "description": "Cancel a booked ticket using its unique ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "unique_id": {
                "type": "string",
                "description": "The unique ID of the ticket to be cancelled",
            },
        },
        "required": ["unique_id"],
        "additionalProperties": False
    }
}

show_bookings_function = {
    "name": "show_bookings",
    "description": "Show all bookings for a specific user.",
    "parameters": {
        "type": "object",
        "properties": {
            "user": {
                "type": "string",
                "description": "The name of the user whose bookings are to be shown",
            },
        },
        "required": ["user"],
        "additionalProperties": False
    }
}


tools = [{"type": "function", "function": get_ticket_price_function}, {"type": "function", "function": set_ticket_price_function}, {"type": "function", "function": book_ticket_function}, {"type": "function", "function": cancel_ticket_function}, {"type": "function", "function": show_bookings_function}]

In [4]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image

In [5]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            # cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": f"Price for {city} set to ${price}",
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "book_ticket":
            arguments = json.loads(tool_call.function.arguments)
            user = arguments.get('user')
            govt_id = arguments.get('govt_id')
            source = arguments.get('source')
            destination = arguments.get('destination')
            date = arguments.get('date')
            time = arguments.get('time')
            booking_response = book_ticket(user, govt_id, source, destination, date, time)
            responses.append({
                "role": "tool",
                "content": booking_response,
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "cancel_ticket":
            arguments = json.loads(tool_call.function.arguments)
            unique_id = arguments.get('unique_id')
            cancellation_response = cancel_ticket(unique_id)
            responses.append({
                "role": "tool",
                "content": cancellation_response,
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "show_bookings":
            arguments = json.loads(tool_call.function.arguments)
            user = arguments.get('user')
            bookings_response = show_bookings(user)
            responses.append({
                "role": "tool",
                "content": bookings_response,
                "tool_call_id": tool_call.id
            })

    return responses, cities

In [6]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


c:\Users\Ashutosh Thakur\OneDrive\Desktop\Ed_llm\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py:1816: UserWarning: A function (chat) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [chatbot, audio]
    Output values returned:
        [[{'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello! How can I assist you with your flight today?'}], b'\xff\xf3\xc4\xc4\x00[<9\x98\x01Y\xc0\x002\x8a2\x081\x840\x80AH\x15\x0c\xccSM4N\x07\r\xa6\rD\x8d%\r\x86\x8d\xe8\x0e\x07\x8d\xa5\x00J =w\xac:\x81\xa1,\xb2\xe5\x93.;~\xd6\xdcy"\x98\x17p\xb2\x80\x00\x18\x04d\x01\xac\xe6\x94\x99\xcee\x19h\xd3\r\xc7\xb7\x9d\x8c?\x0c*J\x1d\x87!\x9d\xa9\x9a\x12\xd0\x0e\x83\xean\xe3\xcb\xe9\xe3\x14\x91\x87a\x86,"\xa4b\x11J\x92\xcaID0\xec9\x0f\xe4\xe7\xd7\x86\xe1\xfa\xce\x02\xc2*E\x88\xa0\xeb\xbd\xaf\xc3\xfd\xa4\x8cC\x0eC\x13k\xf0\xfcn6\xff\xbf\xef\xfc\xbf\x9b\xcf<\xffq6v\x98\x89\x88\xb1\x18\x83\\\x87/>\xec=\x15\x10\x08\\\x82\xe4 \x1